In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!ls "/content/drive/MyDrive"

 03_23521129_TranTuyetNhu
 09_23521129_23521499
 09_23.gdoc
'19522305_Le Ngoc Minh Thu.gdoc'
'20260720_チケットのステータス変更履歴 (2) (1).gsheet'
'20260720_チケットのステータス変更履歴 (2).gsheet'
'20260720_チケットのステータス変更履歴 (4).gsheet'
'20260720_チケットのステータス変更履歴 (5) (1).gsheet'
'20260720_チケットのステータス変更履歴 (5).gsheet'
'20260720_チケットのステータス変更履歴 (6) (1).gsheet'
'20260720_チケットのステータス変更履歴 (6) (2).gsheet'
'20260720_チケットのステータス変更履歴 (6).csv'
'20260720_チケットのステータス変更履歴 (6).gsheet'
'20260720_チケットのステータス変更履歴 (7).gsheet'
'20260720_チケットのステータス変更履歴 (8).gsheet'
 20260720_チケットのステータス変更履歴.gsheet
'20260721_チケットのステータス変更履歴 (10).gsheet'
'20260721_チケットのステータス変更履歴 (6).gsheet'
'20260721_チケットのステータス変更履歴 (7).gsheet'
'20260721_チケットのステータス変更履歴 (8).gsheet'
'20260721_チケットのステータス変更履歴 (9).gsheet'
 23_23521129_23521499
 23521129_TuyetNhu_BCThucTap.docx
'Android Feedback Template (1).gdoc'
'Android Feedback Template.gdoc'
 applicants_20261005_161826.gsheet
'Báo cáo đồ án (1) (1).docx'
'Báo cáo đồ án (1).docx'
 Bao_cao_Lab2_Lam_quen_voi_Layout.gdoc
 Bao_

In [3]:
BASE_DIR = "/content/drive/MyDrive/MXH"
PROCESSED_DIR = f"{BASE_DIR}/processed"

In [4]:
import os

print(os.listdir(PROCESSED_DIR))

['honduras_bad_normalized.jsonl', 'honduras_good_normalized.jsonl', 'quality_audit', 'honduras_account_splits.csv', 'fox8_account_manifest.csv', 'fox8_account_splits.csv', 'fox8_train_ids.csv', 'fox8_test_ids.csv', 'honduras_validation_ids.csv', 'honduras_train_ids.csv', 'fox8_validation_ids.csv', 'honduras_test_ids.csv', 'honduras_account_manifest.csv']


In [5]:
import json
import hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

In [6]:
!pip -q install pyarrow

In [7]:
manifest_path = f"{PROCESSED_DIR}/honduras_account_manifest.csv"
splits_path = f"{PROCESSED_DIR}/honduras_account_splits.csv"

manifest = pd.read_csv(
    manifest_path,
    dtype={"user_id": str}
)

splits = pd.read_csv(
    splits_path,
    dtype={"user_id": str}
)

In [8]:
print("Manifest shape:", manifest.shape)
print("Splits shape:", splits.shape)

display(manifest.head())
display(splits.head())

Manifest shape: (224685, 10)
Splits shape: (224685, 12)


,user_id,label,tweet_count,bad_file_tweet_count,good_file_tweet_count,missing_tweetid_count,missing_text_count,missing_time_count,label_conflict,label_values
0,1000005746078216193,1,1,0,1,0,0,0,False,1
1,1000006078409650177,1,1,0,1,0,0,0,False,1
2,100001435,1,1,0,1,0,0,0,False,1
3,100001645,1,1,0,1,0,0,0,False,1
4,1000017357534883840,1,1,0,1,0,0,0,False,1


,user_id,label,tweet_count,bad_file_tweet_count,good_file_tweet_count,missing_tweetid_count,missing_text_count,missing_time_count,label_conflict,label_values,split,dataset_name
0,884526169827180545,1,6,0,6,0,0,0,False,1,train,Honduras
1,111301873,1,13,0,13,0,0,0,False,1,train,Honduras
2,138430986,1,4,0,4,0,0,0,False,1,train,Honduras
3,2339444377,1,5,0,5,0,0,0,False,1,train,Honduras
4,1189671918,1,1,0,1,0,0,0,False,1,train,Honduras


**Xác minh ý nghĩa good = 0/1**

In [9]:
label_summary = (
    manifest
    .groupby("label")
    .agg(
        account_count=("user_id", "count"),
        tweet_count=("tweet_count", "sum"),
        bad_tweets=("bad_file_tweet_count", "sum"),
        good_tweets=("good_file_tweet_count", "sum"),
    )
)

display(label_summary)

,account_count,tweet_count,bad_tweets,good_tweets
label,,,,
0,1866,137252,137252,0
1,222819,1125578,0,1125578


In [10]:
assert (
    manifest.loc[manifest["label"] == 0, "good_file_tweet_count"].sum()
    == 0
)

assert (
    manifest.loc[manifest["label"] == 1, "bad_file_tweet_count"].sum()
    == 0
)

print("PASS: label mapping is consistent with bad/good source files.")

PASS: label mapping is consistent with bad/good source files.


Label validation:
- label = 0 corresponds to Bad / malicious Honduras data.
- label = 1 corresponds to Good / genuine Honduras data.
- No cross-source label inconsistency was observed in the account manifest.

**Xác minh đơn vị nhãn**

In [11]:
print("Total accounts:", manifest["user_id"].nunique())

print(
    "Label conflict accounts:",
    manifest["label_conflict"].sum()
)

Total accounts: 224685
Label conflict accounts: 0


In [12]:
display(
    manifest["label_values"]
    .value_counts(dropna=False)
)

,count
label_values,
1,222819
0,1866


In [13]:
assert manifest["user_id"].is_unique
assert manifest["label_conflict"].sum() == 0

print("PASS: one account has one consistent label.")

PASS: one account has one consistent label.


Dữ liệu tweet có thể được aggregate theo user_id để tạo mẫu account-level vì không phát hiện account nào chứa đồng thời nhãn Bad và Good trong manifest hiện tại.

**Kiểm tra account ID**

In [14]:
print("Rows:", len(manifest))
print("Unique user IDs:", manifest["user_id"].nunique())
print("Missing user IDs:", manifest["user_id"].isna().sum())
print("Duplicate user IDs:", manifest["user_id"].duplicated().sum())

Rows: 224685
Unique user IDs: 224685
Missing user IDs: 0
Duplicate user IDs: 0


In [15]:
assert manifest["user_id"].notna().all()
assert manifest["user_id"].is_unique

print("PASS: account IDs are unique and non-missing.")

PASS: account IDs are unique and non-missing.


**Kiểm tra missing values**

In [16]:
missing_summary = pd.DataFrame({
    "field": [
        "tweet_id",
        "text",
        "timestamp"
    ],
    "missing_records": [
        manifest["missing_tweetid_count"].sum(),
        manifest["missing_text_count"].sum(),
        manifest["missing_time_count"].sum(),
    ],
    "accounts_affected": [
        (manifest["missing_tweetid_count"] > 0).sum(),
        (manifest["missing_text_count"] > 0).sum(),
        (manifest["missing_time_count"] > 0).sum(),
    ]
})

display(missing_summary)

,field,missing_records,accounts_affected
0,tweet_id,0,0
1,text,0,0
2,timestamp,0,0


**Kiểm tra duplicate ở account level**

In [17]:
duplicate_accounts = manifest[
    manifest["user_id"].duplicated(keep=False)
]

print("Duplicate account rows:", len(duplicate_accounts))

Duplicate account rows: 0


**Kiểm tra duplicate tweet audit**

In [18]:
AUDIT_DIR = f"{PROCESSED_DIR}/quality_audit"

exact_dup = pd.read_csv(
    f"{AUDIT_DIR}/group_a_exact_duplicate_candidates.csv",
    dtype={"tweet_id": str}
)

metadata_conflicts = pd.read_csv(
    f"{AUDIT_DIR}/group_b_metadata_conflicts.csv",
    dtype={"tweet_id": str}
)

text_conflicts = pd.read_csv(
    f"{AUDIT_DIR}/group_c_text_conflicts.csv",
    dtype={"tweet_id": str}
)

In [19]:
print("Exact duplicate candidates:", len(exact_dup))
print("Metadata conflicts:", len(metadata_conflicts))
print("Text conflicts:", len(text_conflicts))

Exact duplicate candidates: 5259
Metadata conflicts: 4
Text conflicts: 385


Tweet-ID audit identified:
- 5,259 exact duplicate candidates,
- 4 metadata-conflict IDs,
- 385 text-conflict IDs.

Only exact duplicates may be considered for deterministic deduplication.
Conflicting records must remain flagged for review rather than being
silently removed.

**Kiểm tra split**

In [20]:
display(
    splits["split"]
    .value_counts()
)

,count
split,
train,157279
test,44938
validation,22468


In [21]:
split_label_table = pd.crosstab(
    splits["split"],
    splits["label"]
)

display(split_label_table)

label,0,1
split,,
test,373,44565
train,1306,155973
validation,187,22281


In [22]:
split_stats = (
    splits.groupby("split")
    .agg(
        accounts=("user_id", "count"),
        bad_accounts=("label", lambda x: (x == 0).sum()),
        good_accounts=("label", lambda x: (x == 1).sum()),
    )
)

split_stats["bad_ratio"] = (
    split_stats["bad_accounts"]
    / split_stats["accounts"]
)

display(split_stats)

,accounts,bad_accounts,good_accounts,bad_ratio
split,,,,
test,44938,373,44565,0.008300
train,157279,1306,155973,0.008304
validation,22468,187,22281,0.008323


**Kiểm tra leakage giữa split**

In [23]:
split_count_per_user = (
    splits.groupby("user_id")["split"]
    .nunique()
)

leaked_users = split_count_per_user[
    split_count_per_user > 1
]

print(
    "Accounts appearing in multiple splits:",
    len(leaked_users)
)

Accounts appearing in multiple splits: 0


In [24]:
assert len(leaked_users) == 0

print("PASS: no account-level split leakage.")

PASS: no account-level split leakage.


**Đối chiếu 3 file ID với account_splits**

In [25]:
train_ids = pd.read_csv(
    f"{PROCESSED_DIR}/honduras_train_ids.csv",
    dtype={"user_id": str}
)

val_ids = pd.read_csv(
    f"{PROCESSED_DIR}/honduras_validation_ids.csv",
    dtype={"user_id": str}
)

test_ids = pd.read_csv(
    f"{PROCESSED_DIR}/honduras_test_ids.csv",
    dtype={"user_id": str}
)

In [26]:
print("train IDs:", len(train_ids))
print("validation IDs:", len(val_ids))
print("test IDs:", len(test_ids))

train IDs: 157279
validation IDs: 22468
test IDs: 44938


In [27]:
expected_train = set(
    splits.loc[
        splits["split"] == "train",
        "user_id"
    ]
)

expected_val = set(
    splits.loc[
        splits["split"] == "validation",
        "user_id"
    ]
)

expected_test = set(
    splits.loc[
        splits["split"] == "test",
        "user_id"
    ]
)

actual_train = set(train_ids["user_id"])
actual_val = set(val_ids["user_id"])
actual_test = set(test_ids["user_id"])

print(
    "Train match:",
    actual_train == expected_train
)

print(
    "Validation match:",
    actual_val == expected_val
)

print(
    "Test match:",
    actual_test == expected_test
)

Train match: True
Validation match: True
Test match: True


In [28]:
assert actual_train == expected_train
assert actual_val == expected_val
assert actual_test == expected_test

print("PASS: split ID files match account_splits.csv.")

PASS: split ID files match account_splits.csv.


**Đối chiếu split với manifest**

In [29]:
manifest_ids = set(manifest["user_id"])
split_ids = set(splits["user_id"])

print(
    "IDs in split but not manifest:",
    len(split_ids - manifest_ids)
)

print(
    "IDs in manifest but not split:",
    len(manifest_ids - split_ids)
)

IDs in split but not manifest: 0
IDs in manifest but not split: 0


In [30]:
assert split_ids == manifest_ids

print("PASS: manifest and split contain the same account IDs.")

PASS: manifest and split contain the same account IDs.


**Kiểm tra label giữa manifest và split có bị đổi không**

In [31]:
check = splits.merge(
    manifest[
        [
            "user_id",
            "label"
        ]
    ],
    on="user_id",
    how="left",
    suffixes=("_split", "_manifest")
)

label_mismatch = check[
    check["label_split"]
    != check["label_manifest"]
]

print(
    "Label mismatch:",
    len(label_mismatch)
)

Label mismatch: 0


In [32]:
assert len(label_mismatch) == 0

print("PASS: labels are preserved across manifest and split.")

PASS: labels are preserved across manifest and split.


**Tạo bảng summary tạm thời**

In [33]:
summary_rows = [
    ["total_accounts", len(manifest)],
    [
        "bad_accounts_label_0",
        (manifest["label"] == 0).sum()
    ],
    [
        "good_accounts_label_1",
        (manifest["label"] == 1).sum()
    ],
    [
        "total_tweets",
        manifest["tweet_count"].sum()
    ],
    [
        "bad_tweets",
        manifest["bad_file_tweet_count"].sum()
    ],
    [
        "good_tweets",
        manifest["good_file_tweet_count"].sum()
    ],
    [
        "duplicate_account_ids",
        manifest["user_id"].duplicated().sum()
    ],
    [
        "label_conflict_accounts",
        manifest["label_conflict"].sum()
    ],
    [
        "missing_tweet_id",
        manifest["missing_tweetid_count"].sum()
    ],
    [
        "missing_text",
        manifest["missing_text_count"].sum()
    ],
    [
        "missing_timestamp",
        manifest["missing_time_count"].sum()
    ],
    [
        "accounts_in_multiple_splits",
        len(leaked_users)
    ],
    [
        "exact_duplicate_tweet_candidates",
        len(exact_dup)
    ],
    [
        "metadata_conflict_tweet_ids",
        len(metadata_conflicts)
    ],
    [
        "text_conflict_tweet_ids",
        len(text_conflicts)
    ],
]

validation_summary = pd.DataFrame(
    summary_rows,
    columns=["metric", "value"]
)

display(validation_summary)

,metric,value
0,total_accounts,224685
1,bad_accounts_label_0,1866
2,good_accounts_label_1,222819
3,total_tweets,1262830
4,bad_tweets,137252
5,good_tweets,1125578
6,duplicate_account_ids,0
7,label_conflict_accounts,0
8,missing_tweet_id,0
9,missing_text,0


## Honduras data & label validation – preliminary conclusion

- The current processed dataset contains 224,685 unique accounts and
  1,262,830 tweets.
- Label mapping is consistent with the source files:
  - 0 = Bad / malicious IO-related data
  - 1 = Good / genuine data
- No account-level label conflict was found.
- No missing tweet ID, text, or timestamp is reported by the current
  account manifest.
- Existing account splits contain 157,279 train, 22,468 validation,
  and 44,938 test accounts.
- No account occurs in more than one split.
- The dataset is highly imbalanced: 1,866 Bad vs 222,819 Good accounts.
- Tweet-ID audit currently contains exact duplicate candidates and a
  smaller number of metadata/text conflicts; conflicting records are
  retained for further review.
- Therefore, Honduras can be safely aggregated at account level using
  the existing account IDs and split assignments, subject to the
  duplicate-handling policy used in later preprocessing.

# Part 2 — Account-level Feature Construction

Mục tiêu:
- Aggregate normalized Honduras tweets thành account-level features.
- Giữ nguyên account ID, label và split đã xác minh.
- Không trộn train/validation/test.
- Tạo các feature đơn giản: tweet count, average text length,
  URL/hashtag/mention ratio và retweet ratio.

In [34]:
BAD_JSONL = f"{PROCESSED_DIR}/honduras_bad_normalized.jsonl"
GOOD_JSONL = f"{PROCESSED_DIR}/honduras_good_normalized.jsonl"

SPLITS_PATH = f"{PROCESSED_DIR}/honduras_account_splits.csv"

AUDIT_DIR = f"{PROCESSED_DIR}/quality_audit"

FEATURE_CSV_PATH = f"{PROCESSED_DIR}/honduras_features.csv"
FEATURE_PARQUET_PATH = f"{PROCESSED_DIR}/honduras_features.parquet"

QUALITY_SUMMARY_PATH = f"{PROCESSED_DIR}/data_quality_summary.csv"

print(BAD_JSONL)
print(GOOD_JSONL)

/content/drive/MyDrive/MXH/processed/honduras_bad_normalized.jsonl
/content/drive/MyDrive/MXH/processed/honduras_good_normalized.jsonl


In [35]:
import json

def show_first_record(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)

                print("Keys:")
                print(obj.keys())

                print("\nSample:")
                for key in [
                    "source",
                    "account_id",
                    "tweet_id",
                    "label",
                    "text",
                    "created_at",
                    "hashtags",
                    "urls",
                    "user_mentions",
                    "retweet_tweet_id",
                    "retweet_user_id",
                ]:
                    print(f"{key}: {obj.get(key)}")

                break

show_first_record(BAD_JSONL)

Keys:
dict_keys(['source', 'account_id', 'tweet_id', 'label', 'text', 'created_at', 'language', 'hashtags', 'urls', 'user_mentions', 'retweet_tweet_id', 'retweet_user_id', 'reply_tweet_id', 'reply_user_id', 'quoted_tweet_id', 'follower_count', 'following_count', 'like_count', 'retweet_count', 'reply_count', 'quote_count', 'account_creation_date'])

Sample:
source: honduras_bad
account_id: 1204524215807942656
tweet_id: 1208055943281418241
label: 0
text: RT @JuanOrlandoH: Los nombres de estos municipios de Copán e Intibucá resumen lo que es la Navidad: Unión, Dulce, Esperanza, Jesús... ¡Siga…
created_at: 1576857900000
hashtags: []
urls: []
user_mentions: ['58244743']
retweet_tweet_id: None
retweet_user_id: None


## Feature definitions

Mỗi dòng output tương ứng với một account.

- `tweet_count`: số tweet của account sau khi xử lý exact duplicates.
- `avg_text_length`: trung bình số ký tự của text.
- `url_ratio`: tỷ lệ tweet của account có ít nhất một URL.
- `hashtag_ratio`: tỷ lệ tweet có ít nhất một hashtag.
- `mention_ratio`: tỷ lệ tweet có ít nhất một user mention.
- `retweet_ratio`: tỷ lệ tweet được nhận diện là retweet.

Retweet được nhận diện nếu:
1. `retweet_tweet_id` hoặc `retweet_user_id` tồn tại; hoặc
2. text bắt đầu bằng dạng chuẩn `RT @`.

Exact duplicates được chỉ giữ một bản ghi.
Metadata/text conflict records chưa tự động loại bỏ ở bước này.

**Xử lý duplicate trước khi aggregate**

In [36]:
exact_dup = pd.read_csv(
    f"{AUDIT_DIR}/group_a_exact_duplicate_candidates.csv",
    dtype={"tweet_id": str}
)

metadata_conflicts = pd.read_csv(
    f"{AUDIT_DIR}/group_b_metadata_conflicts.csv",
    dtype={"tweet_id": str}
)

text_conflicts = pd.read_csv(
    f"{AUDIT_DIR}/group_c_text_conflicts.csv",
    dtype={"tweet_id": str}
)

exact_duplicate_ids = set(
    exact_dup["tweet_id"].astype(str)
)

metadata_conflict_ids = set(
    metadata_conflicts["tweet_id"].astype(str)
)

text_conflict_ids = set(
    text_conflicts["tweet_id"].astype(str)
)

print("Exact duplicate IDs:", len(exact_duplicate_ids))
print("Metadata conflict IDs:", len(metadata_conflict_ids))
print("Text conflict IDs:", len(text_conflict_ids))

Exact duplicate IDs: 5259
Metadata conflict IDs: 4
Text conflict IDs: 385


In [37]:
from collections import defaultdict
import re

In [38]:
def is_retweet_record(obj):
    retweet_tweet_id = obj.get("retweet_tweet_id")
    retweet_user_id = obj.get("retweet_user_id")

    if retweet_tweet_id not in (None, "", "null"):
        return 1

    if retweet_user_id not in (None, "", "null"):
        return 1

    text = obj.get("text") or ""

    if re.match(r"^\s*RT\s+@", text, flags=re.IGNORECASE):
        return 1

    return 0

In [39]:
account_stats = defaultdict(lambda: {
    "label": None,

    "tweet_count": 0,
    "text_length_sum": 0,

    "url_tweet_count": 0,
    "hashtag_tweet_count": 0,
    "mention_tweet_count": 0,
    "retweet_tweet_count": 0,
})

raw_record_count = 0
kept_record_count = 0
skipped_exact_duplicate_count = 0

seen_exact_duplicate_ids = set()

In [40]:
def process_jsonl(path):

    global raw_record_count
    global kept_record_count
    global skipped_exact_duplicate_count

    with open(path, "r", encoding="utf-8") as f:

        for line in f:

            if not line.strip():
                continue

            obj = json.loads(line)

            raw_record_count += 1

            account_id = str(obj["account_id"])
            tweet_id = str(obj["tweet_id"])
            label = int(obj["label"])

            # ---------------------------------
            # Exact duplicate handling
            # ---------------------------------
            if tweet_id in exact_duplicate_ids:

                if tweet_id in seen_exact_duplicate_ids:
                    skipped_exact_duplicate_count += 1
                    continue

                seen_exact_duplicate_ids.add(tweet_id)

            # ---------------------------------
            # Account label consistency
            # ---------------------------------
            stats = account_stats[account_id]

            if stats["label"] is None:
                stats["label"] = label

            elif stats["label"] != label:
                raise ValueError(
                    f"Label conflict for account {account_id}: "
                    f"{stats['label']} vs {label}"
                )

            # ---------------------------------
            # Basic tweet information
            # ---------------------------------
            text = obj.get("text") or ""

            hashtags = obj.get("hashtags") or []
            urls = obj.get("urls") or []
            mentions = obj.get("user_mentions") or []

            # ---------------------------------
            # Aggregate
            # ---------------------------------
            stats["tweet_count"] += 1
            stats["text_length_sum"] += len(text)

            stats["url_tweet_count"] += int(
                len(urls) > 0
            )

            stats["hashtag_tweet_count"] += int(
                len(hashtags) > 0
            )

            stats["mention_tweet_count"] += int(
                len(mentions) > 0
            )

            stats["retweet_tweet_count"] += (
                is_retweet_record(obj)
            )

            kept_record_count += 1

**Chạy trên Bad + Good**

In [41]:
process_jsonl(BAD_JSONL)

print("Finished BAD")

process_jsonl(GOOD_JSONL)

print("Finished GOOD")

Finished BAD
Finished GOOD


In [42]:
print("Raw records:", raw_record_count)
print("Kept records:", kept_record_count)
print(
    "Skipped exact duplicates:",
    skipped_exact_duplicate_count
)

print(
    "Accounts aggregated:",
    len(account_stats)
)

Raw records: 1262830
Kept records: 1255548
Skipped exact duplicates: 7282
Accounts aggregated: 224685


**Chuyển accumulator thành DataFrame**

In [43]:
feature_rows = []

for account_id, stats in account_stats.items():

    n = stats["tweet_count"]

    feature_rows.append({
        "account_id": account_id,

        "label_from_json": stats["label"],

        "tweet_count": n,

        "avg_text_length": (
            stats["text_length_sum"] / n
            if n > 0 else 0
        ),

        "url_ratio": (
            stats["url_tweet_count"] / n
            if n > 0 else 0
        ),

        "hashtag_ratio": (
            stats["hashtag_tweet_count"] / n
            if n > 0 else 0
        ),

        "mention_ratio": (
            stats["mention_tweet_count"] / n
            if n > 0 else 0
        ),

        "retweet_ratio": (
            stats["retweet_tweet_count"] / n
            if n > 0 else 0
        ),
    })

features = pd.DataFrame(feature_rows)

print(features.shape)

display(features.head())

(224685, 8)


,account_id,label_from_json,tweet_count,avg_text_length,url_ratio,hashtag_ratio,mention_ratio,retweet_ratio
0,1204524215807942656,0,58,150.896552,0.017241,0.103448,1.00,0.431034
1,1198721272496246784,0,23,137.478261,0.043478,0.260870,1.00,0.956522
2,1195492958100959232,0,25,132.160000,0.040000,0.200000,0.92,0.920000
3,1191504807724552192,0,98,138.826531,0.051020,0.255102,1.00,0.989796
4,1179467160676192256,0,16,153.937500,0.062500,0.062500,1.00,0.750000


**Gắn label + split từ file đã xác minh**

In [44]:
splits = pd.read_csv(
    SPLITS_PATH,
    dtype={"user_id": str}
)

split_map = splits[
    [
        "user_id",
        "label",
        "split"
    ]
].copy()

split_map = split_map.rename(
    columns={"user_id": "account_id"}
)

In [45]:
features = features.merge(
    split_map,
    on="account_id",
    how="left",
    validate="one_to_one"
)

In [46]:
print(features.shape)

display(
    features[
        [
            "account_id",
            "label_from_json",
            "label",
            "split"
        ]
    ].head()
)

(224685, 10)


,account_id,label_from_json,label,split
0,1204524215807942656,0,0,train
1,1198721272496246784,0,0,train
2,1195492958100959232,0,0,train
3,1191504807724552192,0,0,test
4,1179467160676192256,0,0,train


In [47]:
label_mismatch = features[
    features["label_from_json"]
    != features["label"]
]

print(
    "Label mismatches:",
    len(label_mismatch)
)

Label mismatches: 0


In [48]:
assert len(label_mismatch) == 0

print(
    "PASS: JSON labels match manifest/split labels."
)

PASS: JSON labels match manifest/split labels.


**Kiểm tra account không bị mất**

In [49]:
feature_ids = set(
    features["account_id"]
)

split_ids = set(
    split_map["account_id"]
)

print(
    "Feature accounts:",
    len(feature_ids)
)

print(
    "Split accounts:",
    len(split_ids)
)

print(
    "Missing from features:",
    len(split_ids - feature_ids)
)

print(
    "Extra in features:",
    len(feature_ids - split_ids)
)

Feature accounts: 224685
Split accounts: 224685
Missing from features: 0
Extra in features: 0


In [50]:
assert feature_ids == split_ids

print(
    "PASS: feature table contains exactly all manifest accounts."
)

PASS: feature table contains exactly all manifest accounts.


**Kiểm tra phân bố split**

In [51]:
display(
    features["split"]
    .value_counts()
)

,count
split,
train,157279
test,44938
validation,22468


In [52]:
display(
    pd.crosstab(
        features["split"],
        features["label"]
    )
)

label,0,1
split,,
test,373,44565
train,1306,155973
validation,187,22281


**Validate feature values**

In [53]:
ratio_columns = [
    "url_ratio",
    "hashtag_ratio",
    "mention_ratio",
    "retweet_ratio"
]

for col in ratio_columns:

    assert features[col].between(
        0, 1
    ).all()

print(
    "PASS: all ratio features are within [0, 1]."
)

PASS: all ratio features are within [0, 1].


In [54]:
assert (
    features["tweet_count"] > 0
).all()

print(
    "PASS: all accounts contain at least one tweet."
)

PASS: all accounts contain at least one tweet.


In [55]:
required_cols = [
    "account_id",
    "label",
    "split",
    "tweet_count",
    "avg_text_length",
    "url_ratio",
    "hashtag_ratio",
    "mention_ratio",
    "retweet_ratio",
]

display(
    features[required_cols]
    .isna()
    .sum()
)

,0
account_id,0
label,0
split,0
tweet_count,0
avg_text_length,0
url_ratio,0
hashtag_ratio,0
mention_ratio,0
retweet_ratio,0


**Xóa cột kiểm tra tạm**

In [56]:
features = features.drop(
    columns=["label_from_json"]
)

In [57]:
features = features[
    [
        "account_id",
        "label",
        "split",
        "tweet_count",
        "avg_text_length",
        "url_ratio",
        "hashtag_ratio",
        "mention_ratio",
        "retweet_ratio",
    ]
]

In [58]:
display(features.head())

print(features.shape)

,account_id,label,split,tweet_count,avg_text_length,url_ratio,hashtag_ratio,mention_ratio,retweet_ratio
0,1204524215807942656,0,train,58,150.896552,0.017241,0.103448,1.00,0.431034
1,1198721272496246784,0,train,23,137.478261,0.043478,0.260870,1.00,0.956522
2,1195492958100959232,0,train,25,132.160000,0.040000,0.200000,0.92,0.920000
3,1191504807724552192,0,test,98,138.826531,0.051020,0.255102,1.00,0.989796
4,1179467160676192256,0,train,16,153.937500,0.062500,0.062500,1.00,0.750000


(224685, 9)


**Xem thống kê feature**

In [59]:
display(
    features[
        [
            "tweet_count",
            "avg_text_length",
            "url_ratio",
            "hashtag_ratio",
            "mention_ratio",
            "retweet_ratio",
        ]
    ].describe()
)

,tweet_count,avg_text_length,url_ratio,hashtag_ratio,mention_ratio,retweet_ratio
count,224685.000000,224685.000000,224685.000000,224685.000000,224685.00000,224685.000000
mean,5.588037,142.040059,0.167388,0.868598,0.91601,0.914241
std,59.197478,24.616567,0.335964,0.287881,0.25686,0.267953
min,1.000000,2.000000,0.000000,0.000000,0.00000,0.000000
25%,1.000000,139.454545,0.000000,0.933333,1.00000,1.000000
50%,1.000000,140.000000,0.000000,1.000000,1.00000,1.000000
75%,3.000000,140.000000,0.083333,1.000000,1.00000,1.000000
max,21695.000000,450.000000,1.000000,1.000000,1.00000,1.000000


In [60]:
feature_by_label = (
    features
    .groupby("label")
    [
        [
            "tweet_count",
            "avg_text_length",
            "url_ratio",
            "hashtag_ratio",
            "mention_ratio",
            "retweet_ratio",
        ]
    ]
    .mean()
)

display(feature_by_label)

,tweet_count,avg_text_length,url_ratio,hashtag_ratio,mention_ratio,retweet_ratio
label,,,,,,
0,73.554126,129.753985,0.062737,0.180652,0.939027,0.723737
1,5.018854,142.142949,0.168265,0.874360,0.915817,0.915836


In [61]:
feature_quality_rows = pd.DataFrame(
    [
        ["feature_table_accounts", len(features)],
        ["raw_normalized_tweet_records", raw_record_count],
        ["feature_tweet_records_after_exact_dedup", kept_record_count],
        ["exact_duplicate_records_skipped", skipped_exact_duplicate_count],
        ["feature_label_mismatches", len(label_mismatch)],
        ["feature_accounts_missing_from_split", len(split_ids - feature_ids)],
        ["feature_accounts_extra_vs_split", len(feature_ids - split_ids)],
    ],
    columns=["metric", "value"]
)

data_quality_summary = pd.concat(
    [
        validation_summary,
        feature_quality_rows
    ],
    ignore_index=True
)

display(data_quality_summary)

,metric,value
0,total_accounts,224685
1,bad_accounts_label_0,1866
2,good_accounts_label_1,222819
3,total_tweets,1262830
4,bad_tweets,137252
5,good_tweets,1125578
6,duplicate_account_ids,0
7,label_conflict_accounts,0
8,missing_tweet_id,0
9,missing_text,0


**Export feature table**

In [62]:
import os

print("PROCESSED_DIR:", PROCESSED_DIR)
print("Exists:", os.path.exists(PROCESSED_DIR))
print("Writable:", os.access(PROCESSED_DIR, os.W_OK))

PROCESSED_DIR: /content/drive/MyDrive/MXH/processed
Exists: True
Writable: True


In [63]:
TEST_PATH = f"{PROCESSED_DIR}/__write_test__.txt"

try:
    with open(TEST_PATH, "w", encoding="utf-8") as f:
        f.write("test")

    print("WRITE TEST: PASS")
    print(TEST_PATH)

    os.remove(TEST_PATH)
    print("DELETE TEST: PASS")

except Exception as e:
    print("WRITE TEST: FAIL")
    print(type(e).__name__, e)

WRITE TEST: FAIL
OSError [Errno 30] Read-only file system: '/content/drive/MyDrive/MXH/processed/__write_test__.txt'


In [64]:
OUTPUT_DIR = "/content/drive/MyDrive/MXH_output"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

FEATURE_CSV_PATH = f"{OUTPUT_DIR}/honduras_features.csv"
FEATURE_PARQUET_PATH = f"{OUTPUT_DIR}/honduras_features.parquet"
QUALITY_SUMMARY_PATH = f"{OUTPUT_DIR}/data_quality_summary.csv"

print("OUTPUT_DIR:", OUTPUT_DIR)

OUTPUT_DIR: /content/drive/MyDrive/MXH_output


In [65]:
TEST_PATH = f"{OUTPUT_DIR}/__write_test__.txt"

with open(TEST_PATH, "w", encoding="utf-8") as f:
    f.write("test")

print("WRITE TEST: PASS")

os.remove(TEST_PATH)

WRITE TEST: PASS


In [66]:
features.to_csv(
    FEATURE_CSV_PATH,
    index=False
)

print(
    "Saved:",
    FEATURE_CSV_PATH
)

Saved: /content/drive/MyDrive/MXH_output/honduras_features.csv


In [67]:
features.to_parquet(
    FEATURE_PARQUET_PATH,
    index=False
)

print(
    "Saved:",
    FEATURE_PARQUET_PATH
)

Saved: /content/drive/MyDrive/MXH_output/honduras_features.parquet


In [68]:
data_quality_summary = pd.concat(
    [
        validation_summary,
        feature_quality_rows
    ],
    ignore_index=True
)

display(data_quality_summary)

,metric,value
0,total_accounts,224685
1,bad_accounts_label_0,1866
2,good_accounts_label_1,222819
3,total_tweets,1262830
4,bad_tweets,137252
5,good_tweets,1125578
6,duplicate_account_ids,0
7,label_conflict_accounts,0
8,missing_tweet_id,0
9,missing_text,0


In [69]:
data_quality_summary.to_csv(
    QUALITY_SUMMARY_PATH,
    index=False
)

print("Saved quality summary:", QUALITY_SUMMARY_PATH)

Saved quality summary: /content/drive/MyDrive/MXH_output/data_quality_summary.csv


**Kiểm tra file đã thật sự tồn tại**

In [70]:
import os

for path in [
    FEATURE_CSV_PATH,
    FEATURE_PARQUET_PATH,
    QUALITY_SUMMARY_PATH,
]:

    print(
        os.path.basename(path),
        "=>",
        os.path.exists(path),
        os.path.getsize(path)
        if os.path.exists(path)
        else None
    )

honduras_features.csv => True 11544718
honduras_features.parquet => True 3792067
data_quality_summary.csv => True 630


## Account-level feature construction – conclusion

- Honduras normalized tweets were aggregated by `account_id`.
- Existing account-level train/validation/test assignments were preserved.
- Labels from normalized JSONL were cross-checked against the authoritative
  split file.
- Exact duplicate tweet candidates were deterministically counted once.
- Metadata/text-conflict records were not automatically removed at this stage.
- The final feature table contains one row per account.
- Basic account-level features include:
  tweet count, average text length, URL ratio, hashtag ratio,
  mention ratio and retweet ratio.
- No account was moved between train/validation/test splits.

# Part 3 — Final QA & Handoff

Mục tiêu:
- Reload các file output đã export.
- Đối chiếu lại account ID, label và split với source manifest.
- Kiểm tra output không bị corruption.
- Tổng hợp các limitation cần bàn giao cho E2/E3.

In [71]:
features_check = pd.read_parquet(
    FEATURE_PARQUET_PATH
)

quality_check = pd.read_csv(
    QUALITY_SUMMARY_PATH
)

print("Feature shape:", features_check.shape)
print("Quality summary shape:", quality_check.shape)

display(features_check.head())
display(quality_check)

Feature shape: (224685, 9)
Quality summary shape: (22, 2)


,account_id,label,split,tweet_count,avg_text_length,url_ratio,hashtag_ratio,mention_ratio,retweet_ratio
0,1204524215807942656,0,train,58,150.896552,0.017241,0.103448,1.00,0.431034
1,1198721272496246784,0,train,23,137.478261,0.043478,0.260870,1.00,0.956522
2,1195492958100959232,0,train,25,132.160000,0.040000,0.200000,0.92,0.920000
3,1191504807724552192,0,test,98,138.826531,0.051020,0.255102,1.00,0.989796
4,1179467160676192256,0,train,16,153.937500,0.062500,0.062500,1.00,0.750000


,metric,value
0,total_accounts,224685
1,bad_accounts_label_0,1866
2,good_accounts_label_1,222819
3,total_tweets,1262830
4,bad_tweets,137252
5,good_tweets,1125578
6,duplicate_account_ids,0
7,label_conflict_accounts,0
8,missing_tweet_id,0
9,missing_text,0


In [72]:
EXPECTED_COLUMNS = [
    "account_id",
    "label",
    "split",
    "tweet_count",
    "avg_text_length",
    "url_ratio",
    "hashtag_ratio",
    "mention_ratio",
    "retweet_ratio",
]

print(features_check.columns.tolist())

assert features_check.columns.tolist() == EXPECTED_COLUMNS

print("PASS: final schema is correct.")

['account_id', 'label', 'split', 'tweet_count', 'avg_text_length', 'url_ratio', 'hashtag_ratio', 'mention_ratio', 'retweet_ratio']
PASS: final schema is correct.


In [73]:
duplicate_accounts = (
    features_check["account_id"]
    .duplicated()
    .sum()
)

print(
    "Duplicate account IDs:",
    duplicate_accounts
)

assert duplicate_accounts == 0

print(
    "PASS: one row corresponds to one account."
)

Duplicate account IDs: 0
PASS: one row corresponds to one account.


In [74]:
final_missing = (
    features_check
    .isna()
    .sum()
)

display(final_missing)

assert (
    final_missing.sum() == 0
)

print(
    "PASS: no missing values in final feature table."
)

,0
account_id,0
label,0
split,0
tweet_count,0
avg_text_length,0
url_ratio,0
hashtag_ratio,0
mention_ratio,0
retweet_ratio,0


PASS: no missing values in final feature table.


In [75]:
final_ids = set(
    features_check["account_id"].astype(str)
)

authoritative_ids = set(
    split_map["account_id"].astype(str)
)

print(
    "Final feature accounts:",
    len(final_ids)
)

print(
    "Authoritative accounts:",
    len(authoritative_ids)
)

print(
    "Missing:",
    len(authoritative_ids - final_ids)
)

print(
    "Extra:",
    len(final_ids - authoritative_ids)
)

assert (
    final_ids == authoritative_ids
)

print(
    "PASS: final feature accounts exactly match split manifest."
)

Final feature accounts: 224685
Authoritative accounts: 224685
Missing: 0
Extra: 0
PASS: final feature accounts exactly match split manifest.


**So sánh label + split từng account**

In [76]:
final_validation = (
    features_check[
        ["account_id", "label", "split"]
    ]
    .merge(
        split_map[
            ["account_id", "label", "split"]
        ],
        on="account_id",
        how="outer",
        suffixes=("_feature", "_manifest"),
        validate="one_to_one"
    )
)

In [77]:
label_diff = final_validation[
    final_validation["label_feature"]
    != final_validation["label_manifest"]
]

split_diff = final_validation[
    final_validation["split_feature"]
    != final_validation["split_manifest"]
]

print(
    "Final label mismatches:",
    len(label_diff)
)

print(
    "Final split mismatches:",
    len(split_diff)
)

assert len(label_diff) == 0
assert len(split_diff) == 0

print(
    "PASS: account labels and splits are preserved exactly."
)

Final label mismatches: 0
Final split mismatches: 0
PASS: account labels and splits are preserved exactly.


In [78]:
handoff_stats = (
    features_check
    .groupby(["split", "label"])
    .agg(
        accounts=("account_id", "count"),
        mean_tweet_count=("tweet_count", "mean"),
        median_tweet_count=("tweet_count", "median"),
        mean_text_length=("avg_text_length", "mean"),
        mean_url_ratio=("url_ratio", "mean"),
        mean_hashtag_ratio=("hashtag_ratio", "mean"),
        mean_mention_ratio=("mention_ratio", "mean"),
        mean_retweet_ratio=("retweet_ratio", "mean"),
    )
    .reset_index()
)

display(handoff_stats)

,split,label,accounts,mean_tweet_count,median_tweet_count,mean_text_length,mean_url_ratio,mean_hashtag_ratio,mean_mention_ratio,mean_retweet_ratio
0,test,0,373,67.453083,34.0,129.338451,0.066593,0.170177,0.941787,0.733990
1,test,1,44565,5.033030,1.0,142.114558,0.170175,0.874664,0.916005,0.914809
2,train,0,1306,74.188361,45.5,129.997497,0.061258,0.185409,0.939231,0.719463
3,train,1,155973,5.041302,1.0,142.112573,0.167706,0.874391,0.915747,0.916060
4,validation,0,187,81.294118,41.0,128.882157,0.065375,0.168320,0.932096,0.733138
5,validation,1,22281,4.833356,1.0,142.412378,0.168354,0.873532,0.915932,0.916320


In [79]:
HANDOFF_STATS_PATH = (
    f"{OUTPUT_DIR}/honduras_feature_profile.csv"
)

handoff_stats.to_csv(
    HANDOFF_STATS_PATH,
    index=False
)

print(
    "Saved:",
    HANDOFF_STATS_PATH
)

Saved: /content/drive/MyDrive/MXH_output/honduras_feature_profile.csv


In [80]:
imbalance_summary = (
    features_check
    .groupby("split")
    .agg(
        total_accounts=("account_id", "count"),
        bad_accounts=(
            "label",
            lambda x: (x == 0).sum()
        ),
        good_accounts=(
            "label",
            lambda x: (x == 1).sum()
        )
    )
)

imbalance_summary["bad_ratio"] = (
    imbalance_summary["bad_accounts"]
    /
    imbalance_summary["total_accounts"]
)

imbalance_summary["good_to_bad_ratio"] = (
    imbalance_summary["good_accounts"]
    /
    imbalance_summary["bad_accounts"]
)

display(imbalance_summary)

,total_accounts,bad_accounts,good_accounts,bad_ratio,good_to_bad_ratio
split,,,,,
test,44938,373,44565,0.008300,119.477212
train,157279,1306,155973,0.008304,119.428025
validation,22468,187,22281,0.008323,119.149733


## Important dataset limitations for downstream experiments

1. Severe class imbalance:
   Bad accounts represent only about 0.83% of the account-level dataset.

2. Strong activity difference:
   Bad accounts have substantially more tweets per account than Good accounts.
   Therefore, `tweet_count` may become a strong shortcut feature.

3. Distribution differences:
   URL, hashtag and retweet usage differ substantially between Bad and Good.
   These differences may reflect behavior, data collection procedure, or both.

4. Evaluation implication:
   Accuracy should not be used as the primary metric.
   E2 should report metrics such as Bad-class F1, precision, recall,
   PR-AUC / Average Precision and Macro-F1.

5. Leakage prevention:
   Existing account-level train/validation/test splits must remain fixed.
   No account may appear in more than one split.

6. Feature engineering:
   Any normalization, scaling, sampling or feature selection must be fitted
   on the training split only.

## Duplicate policy

- 5,259 tweet IDs were identified as exact-duplicate candidates.
- During feature construction, repeated occurrences of those IDs were counted once.
- 7,282 repeated records were skipped in total because some duplicate tweet IDs
  occurred more than twice.
- Metadata-conflict and text-conflict tweet IDs were not automatically removed.
- Their existence is documented in `data_quality_summary.csv` and the quality-audit files.